# Core Level Performance Analysis Example

We aim to classify performance according to the performance analysis lesson. This notebook assumes the repository is available at `/opt/temp-task020` in the single user profile, or that you have cloned it into your home directory directly.

In [ ]:
from pathlib import Path
import subprocess
import platform
import re
import time
import pandas as pd
import matplotlib.pyplot as plt

# Select the benchmark
BENCHMARK_NAME = "N-body CPU"

SOURCE_DIR = Path("/opt/temp-task020/cuda_intro")
EXECUTABLE = SOURCE_DIR / "nbody_cpu"

RESULT_DIR = Path.home() / "performance-work" / "benchmark-results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# Build configuration
BUILD_COMMAND = [
    "g++", "-std=gnu++17", "-O3", "-g",
    "-fno-omit-frame-pointer", "-march=native",
    "nbody.cpp", "-o", "nbody_cpu"
]

# Execution configuration
RUN_COMMAND = [
    str(EXECUTABLE),
    "-n", "1000",
    "--dt", "0.01",
    "--total_time", "0.2",
    "--seed", "42",
    "--quiet",
    "--disable_unit_tests",
]

REPEATS = 5
PIN_CPU = None

print("Benchmark:", BENCHMARK_NAME)
print("Source directory:", SOURCE_DIR)
print("Executable:", EXECUTABLE)
print("Platform:", platform.platform())

In [ ]:
def build_benchmark():
    result = subprocess.run(
        BUILD_COMMAND,
        cwd=SOURCE_DIR,
        text=True,
        capture_output=True
    )

    print(result.stdout)
    print(result.stderr)

    result.check_returncode()

    if not EXECUTABLE.exists():
        raise FileNotFoundError(
            f"Executable not found: {EXECUTABLE}"
        )

    print("Build successful:", EXECUTABLE)


def run_benchmark(affinity_cpu=None):
    cmd = RUN_COMMAND.copy()

    if affinity_cpu is not None:
        cmd = ["taskset", "-c", str(affinity_cpu)] + cmd

    result = subprocess.run(
        cmd,
        cwd=RESULT_DIR,
        text=True,
        capture_output=True
    )

    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        result.check_returncode()

    return result.stdout

## Build N-body

`-O3` enables optimisation; `-g` adds debugging information; `-fno-omit-frame-pointer` can improve stack unwinding in some profilers. `-march=native` targets the CPU visible during the build, so use it only when the executable will run on a compatible node.

In [ ]:
assert (SOURCE_DIR / "nbody.cpp").exists(), f"Cannot find nbody.cpp in {SOURCE_DIR}"

build = subprocess.run(
    ["g++", "-std=gnu++17", "-O3", "-g", "-fno-omit-frame-pointer",
     "-march=native", "nbody.cpp", "-o", "nbody_cpu"],
    cwd=SOURCE_DIR, text=True, capture_output=True
)
print(build.stdout)
print(build.stderr)
build.check_returncode()
print(command_output(["file", str(EXECUTABLE)]))

## Run a correctness

 The sample problem (`N=1000`, `total_time=0.2`)

In [ ]:
def run_nbody(n=1000, total_time=0.2, seed=42, extra_args=None, affinity_cpu=None):
    cmd = [
        str(EXECUTABLE),
        "-n", str(n),
        "--dt", "0.01",
        "--total_time", str(total_time),
        "--seed", str(seed),
        "--quiet",
        "--disable_unit_tests",
    ]
    if extra_args:
        cmd.extend(extra_args)
    if affinity_cpu is not None:
        cmd = ["taskset", "-c", str(affinity_cpu)] + cmd

    completed = subprocess.run(
        cmd, cwd=RESULT_DIR, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    if completed.returncode != 0:
        print(completed.stdout)
        print(completed.stderr)
        completed.check_returncode()
    return completed.stdout

print(run_nbody())

In [ ]:
PIN_CPU = None

N_PARTICLES = 1000
TOTAL_TIME = 0.2
REPEATS = 5

def parse_internal_us(stdout):
    match = re.search(r"Total:\s*([0-9.eE+-]+)\s*us", stdout)
    if not match:
        raise ValueError("Could not find 'Total: ... us' in program output:\n" + stdout)
    return float(match.group(1))

records = []
for repetition in range(REPEATS):
    start = time.perf_counter()
    output = run_nbody(
        n=N_PARTICLES,
        total_time=TOTAL_TIME,
        seed=42,
        affinity_cpu=PIN_CPU,
    )
    wall_seconds = time.perf_counter() - start
    internal_us = parse_internal_us(output)
    records.append({
        "repetition": repetition + 1,
        "internal_seconds": internal_us / 1e6,
        "wall_seconds": wall_seconds,
    })

timings = pd.DataFrame(records)
display(timings)
print(f"Median internal time: {timings.internal_seconds.median():.6f} s")
print(f"Median wall time:     {timings.wall_seconds.median():.6f} s")

## Callgrind

 It is useful for identifying expensive functions and call relationships.

In [ ]:
CALLGRIND_FILE = RESULT_DIR / "callgrind.out"

if command_output(["bash", "-lc", "command -v valgrind"]).startswith("Unavailable"):
    print("Valgrind is not installed in this container.")
else:
    cmd = [
        "valgrind", "--tool=callgrind",
        f"--callgrind-out-file={CALLGRIND_FILE}",
        str(EXECUTABLE),
        "-n", str(N_PARTICLES),
        "--dt", "0.01",
        "--total_time", str(TOTAL_TIME),
        "--seed", "42",
        "--quiet",
        "--disable_unit_tests",
    ]
    result = subprocess.run(cmd, cwd=RESULT_DIR, text=True, capture_output=True)
    print(result.stdout)
    print(result.stderr[-4000:])
    print("Callgrind output exists:", CALLGRIND_FILE.exists())

In [ ]:
ANNOTATE_FILE = RESULT_DIR / "callgrind-annotate.txt"
if CALLGRIND_FILE.exists():
    annotated = subprocess.run(
        ["callgrind_annotate", "--inclusive=yes", str(CALLGRIND_FILE)],
        text=True, capture_output=True, check=True
    )
    ANNOTATE_FILE.write_text(annotated.stdout)
    print("\n".join(annotated.stdout.splitlines()[:70]))
else:
    print("Run the Callgrind cell first.")

## 7. Calculate core performance ratio

- `obtained_peak_gflops`: obtained peak measured for the code 
- `maximum_scalar_peak_gflops`: theoretical scalar peak for one core of the same system

In [ ]:
obtained_peak_gflops = None
maximum_scalar_peak_gflops = None

def classify_core_ratio(obtained, maximum):
    if obtained is None or maximum is None:
        return None, "Enter both performance values first."
    if obtained <= 0 or maximum <= 0:
        raise ValueError("Both GFLOP/s values must be positive.")
    ratio = obtained / maximum
    if ratio >= 0.80:
        label = "Green"
    elif ratio >= 0.60:
        label = "Yellow"
    else:
        label = "Red"
    return ratio, label

ratio, classification = classify_core_ratio(
    obtained_peak_gflops, maximum_scalar_peak_gflops
)
if ratio is None:
    print(classification)
else:
    print(f"Core performance ratio: {ratio:.1%}")
    print(f"Classification: {classification}")

## Rubric practice
- Code A: 34 GFLOP/s
- Code B: 28 GFLOP/s
- Code C: 19 GFLOP/s

In [ ]:
example_maximum = 40.0
examples = pd.DataFrame({
    "Code": ["A", "B", "C"],
    "Obtained GFLOP/s": [34.0, 28.0, 19.0],
})
examples["Core ratio"] = examples["Obtained GFLOP/s"] / example_maximum
examples["Classification"] = examples["Core ratio"].apply(
    lambda x: "Green" if x >= 0.80 else ("Yellow" if x >= 0.60 else "Red")
)
display(examples)